# খরচ দাবি বিশ্লেষণ

এই নোটবুকটি দেখায় কিভাবে এজেন্ট তৈরি করতে হয় যারা প্লাগইন ব্যবহার করে স্থানীয় রসিদ ইমেজ থেকে ভ্রমণ খরচ প্রক্রিয়াজাত করে, একটি খরচ দাবি ইমেল তৈরি করে, এবং একটি পাই চার্ট ব্যবহার করে খরচের ডেটা ভিজ্যুয়ালাইজ করে। এজেন্টরা কাজের প্রেক্ষাপটের উপর ভিত্তি করে গতিশীলভাবে ফাংশন নির্বাচন করে।

ধাপসমূহ:
1. OCR এজেন্ট স্থানীয় রসিদ ইমেজ প্রক্রিয়াজাত করে এবং ভ্রমণ খরচের ডেটা সংগ্রহ করে।
2. ইমেল এজেন্ট একটি খরচ দাবি ইমেল তৈরি করে।

### একটি ভ্রমণ খরচ পরিস্থিতির উদাহরণ:
ধরুন আপনি একজন কর্মচারী যিনি অন্য শহরে একটি ব্যবসায়িক বৈঠকের জন্য ভ্রমণ করছেন। আপনার কোম্পানির নীতি হলো সকল যুক্তিসঙ্গত ভ্রমণ সংশ্লিষ্ট খরচ ফেরত দেওয়া। ভ্রমণ খরচের সম্ভাব্য বিবরণ নিচে দেওয়া হলো:
- পরিবহন:
আপনার বাড়ি শহর থেকে গন্তব্য শহরে যাওয়া এবং ফিরে আসার জন্য বিমান ভাড়া।
বিমানবন্দর যাতায়াতের জন্য ট্যাক্সি বা রাইড-হেইলিং পরিষেবা।
গন্তব্য শহরে স্থানীয় পরিবহন (যেমন পাবলিক ট্রানজিট, ভাড়া করা গাড়ি, বা ট্যাক্সি)।

- বাসস্থান:
বৈঠকের স্থান থেকে নিকটবর্তী একটি মধ্যম মানের ব্যবসায়িক হোটেলে তিন রাত থাকার খরচ।

- খাবার:
সকাল ও দুপুর ও রাতের খাবারের জন্য দৈনিক ভাতা, কোম্পানির দৈনিক নীতির উপর ভিত্তি করে।

- অন্যান্য খরচ:
বিমানবন্দরে পার্কিং খরচ।
হোটেলে ইন্টারনেট ব্যবহার শুল্ক।
টিপস বা ছোট সেবার চার্জ।

- ডকুমেন্টেশন:
আপনি সব রসিদ জমা দেন (ফ্লাইট, ট্যাক্সি, হোটেল, খাবার ইত্যাদি) এবং একটি পূর্ণাঙ্গ খরচ রিপোর্ট ফেরতের জন্য।


## প্রয়োজনীয় লাইব্রেরি আমদানি করুন

নোটবুকের জন্য প্রয়োজনীয় লাইব্রেরি এবং মডিউলগুলি আমদানি করুন।


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## ব্যয় মডেলগুলি সংজ্ঞায়িত করুন

 পৃথক ব্যয়গুলির জন্য একটি Pydantic মডেল তৈরি করুন এবং একটি ExpenseFormatter ক্লাস তৈরি করুন যা ব্যবহারকারীর অনুসন্ধানকে কাঠামোবদ্ধ ব্যয় তথ্যেতে রূপান্তরিত করবে।

 প্রতিটি ব্যয় নিম্নলিখিত ফরম্যাটে উপস্থাপিত হবে:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## টুল সংজ্ঞায়িত করা - ইমেল তৈরি করা

একটি টুল ফাংশন তৈরি করুন যা একটি ব্যয়ের দাবির জন্য ইমেল তৈরি করবে।
- এই টুলটি Microsoft Agent Framework থেকে `@tool` ডেকোরেটর ব্যবহার করে।
- এটি ব্যয়ের মোট পরিমাণ গণনা করে এবং বিশদগুলো ইমেল বডিতে ফরম্যাট করে।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# রসিদ ছবিগুলো থেকে ভ্রমণ খরচ নিষ্কাশনের জন্য টুল

রসিদ ছবিগুলো থেকে ভ্রমণ খরচ নিষ্কাশনের জন্য একটি টুল ফাংশন তৈরি করুন।
- এই টুলটি মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্কের `@tool` ডেকোরেটর ব্যবহার করে।
- এটি রসিদ ছবিটি পড়ে, সেটিকে base64 হিসেবে এনকোড করে, এবং এজেন্ট বিশ্লেষণের জন্য ডাটা URI প্রদান করে।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## প্রক্রিয়াকরণ ব্যয়

এজেন্টগুলি সংজ্ঞায়িত করুন এবং তাদের একটি ক্রমাগত ওয়ার্কফ্লোতে সংযুক্ত করুন `WorkflowBuilder` ব্যবহার করে।
- OCR এজেন্ট রশিদ ছবির থেকে গঠনমূলক ব্যয় তথ্য বের করে `load_receipt_image` টুল ব্যবহার করে।
- ইমেইল এজেন্ট প্রাপ্ত তথ্য নিয়ে একটি পেশাদার ব্যয় দাবি ইমেইল তৈরি করে `generate_expense_email` টুল ব্যবহার করে।
- `WorkflowBuilder` এর `add_edge` দিয়ে একটি ক্রমাগত পাইপলাইন তৈরি করে: OCR এজেন্ট → ইমেইল এজেন্ট।


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## প্রধান ফাংশন

সিরিয়াল ওয়ার্কফ্লো তৈরি করুন এবং চালান যা রসিদ চিত্র প্রক্রিয়াকরণ করে এবং ব্যয় দাবি ইমেল তৈরি করে।


> **নোট:** এই ওয়ার্কফ্লো বর্তমানে রসিদ ছবিটি বেস64 টেক্সট হিসাবে পাস করে, যা বেশিরভাগ চ্যাট মডেল (gpt-4.1-mini সহ) একটি ছবি হিসেবে বিবেচনা করবে না।
> এটি মডেল কন্টেক্সট উইন্ডোর সীমা ছাড়িয়ে যেতে পারে। Azure AI Vision (অথবা অন্য কোনো OCR সরঞ্জাম) দিয়ে OCR চালানো এবং শুধুমাত্র এক্সট্র্যাক্ট করা টেক্সট পাস করা ভালো, অথবা ছবি `image_url` মেসেজ হিসাবে পাঠানোর জন্য রিফ্যাক্টর করুন।
> যদি আপনি শুধুমাত্র কন্টেক্সট এরর এড়াতে চান, তাহলে একটি ছোট রসিদ ছবি ব্যবহার করুন বা বড় কন্টেক্সট উইন্ডো সহ মডেল নির্বাচন করুন।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
